# 01 — Fuentes, limpieza e integración de datos

**Proyecto:** *Del colegio a la universidad: inteligencia territorial para cerrar brechas
de acceso y permanencia en la educación superior de Antioquia.*
Concurso **Datos al Ecosistema 2026 — IA para Colombia** (nivel intermedio).

## Fuentes integradas (5 conjuntos, 4 de datos.gov.co)

| # | Conjunto | Fuente | Filas |
|---|---|---|---|
| 1 | Matriculados UdeA sedes regionales 2026-1 (`URABA 20261.xlsx`) | Universidad de Antioquia | 8.157 |
| 2 | Beneficiarios de programas de acompañamiento ([xk8x-i6kn](https://www.datos.gov.co/d/xk8x-i6kn)) | datos.gov.co — Gobernación de Antioquia | 13.778 |
| 3 | Población Antioquia censada 2018 ([evm3-92yw](https://www.datos.gov.co/d/evm3-92yw)) | datos.gov.co — Gobernación de Antioquia / DANE | 12.825 |
| 4 | Resultados únicos Saber 11, filtro Antioquia 2018+ ([kgxf-xxbe](https://www.datos.gov.co/d/kgxf-xxbe)) | datos.gov.co — ICFES | 297.962 |
| 5 | DIVIPOLA códigos de municipios ([gdxc-w37w](https://www.datos.gov.co/d/gdxc-w37w)) | datos.gov.co — DANE | 1.122 |

Los conjuntos 4 y 5 se descargan por la **API Socrata** con `src/acquire.py`
(el Saber 11 se filtra en el servidor con SoQL: solo colegios de Antioquia y periodos
2018-1 en adelante, 298 mil de 7,1 millones de filas).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

## Problemas de calidad detectados y tratados

1. **Mojibake por doble codificación** en Beneficiarios: `SOÃ‘ARES` → `SOÑARES`,
   `URABÃ` → `URABÁ` (se corrige con `ftfy`).
2. **Categorías inconsistentes**: género `F`/`FEMENINO`/`M`/`MASCULINO`;
   víctima del conflicto `0`/`NO`/`1`/`SI`; subregión `URABA`/`URABÁ`.
3. **Fechas en formatos mixtos** en URABA: celdas datetime y textos `13-APR-1993`.
4. **Códigos de municipio incompatibles**: URABA usa el código DANE *sin* el prefijo
   de departamento (`854` → `05854` Valdivia); se homologa todo a DANE de 5 dígitos.
5. **Centinela `9.99`** en `PROMEDIO_PROGRAMA` para estudiantes sin historia académica
   (se detecta y se excluye de la población del modelo predictivo).

### Evidencia del mojibake (antes → después)

In [2]:
import ftfy
raw = pd.read_excel("../data/raw/Beneficiarios.xlsx",
                    sheet_name="Beneficiarios_de_los_programas_",
                    usecols=["PROGRAMA", "SUBREGIÓN DE RESIDENCIA"])
ejemplos = raw[raw["PROGRAMA"].str.contains("Ã", na=False) |
               raw["SUBREGIÓN DE RESIDENCIA"].str.contains("Ã", na=False)]
antes = ejemplos.drop_duplicates().head(5)
despues = antes.map(ftfy.fix_text)
pd.concat({"antes": antes.reset_index(drop=True),
           "después": despues.reset_index(drop=True)}, axis=1)

antes                           después         
  SUBREGIÓN DE RESIDENCIA  PROGRAMA SUBREGIÓN DE RESIDENCIA PROGRAMA
0                SUROESTE  SOÃ‘ARES                SUROESTE  SOÑARES
1              BAJO CAUCA  SOÃ‘ARES              BAJO CAUCA  SOÑARES
2                   URABA  SOÃ‘ARES                   URABA  SOÑARES
3                NORDESTE  SOÃ‘ARES                NORDESTE  SOÑARES
4                 ORIENTE  SOÃ‘ARES                 ORIENTE  SOÑARES

### Evidencia de los formatos mixtos de fecha

In [3]:
fechas = pd.read_excel("../data/raw/URABA 20261.xlsx",
                       sheet_name="20261_RG_MAT", usecols=["FECH_NACE"])
fechas["tipo_de_dato"] = fechas["FECH_NACE"].map(lambda v: type(v).__name__)
print(fechas["tipo_de_dato"].value_counts())
fechas.groupby("tipo_de_dato").head(2)

tipo_de_dato
datetime    5449
str         2708
Name: count, dtype: int64


,FECH_NACE,tipo_de_dato
0,13-APR-1993,str
1,2001-09-19 00:00:00,datetime
2,1968-05-26 00:00:00,datetime
3,11-APR-1974,str


## Ejecución del pipeline de limpieza e integración

`src/clean.py` normaliza cada fuente y la guarda en parquet; `src/integrate.py`
homologa municipios contra DIVIPOLA (llave: código DANE de 5 dígitos, con auditoría
difusa de nombres vía `rapidfuzz`) y construye las dos matrices analíticas.

In [4]:
import clean
clean.main()

URABA: 8157 filas | fechas no parseadas: 0 | sin cod DANE vive: 1 | estrato faltante: 324


TypeError: numpy string dtypes are not allowed, use 'str' or 'object' instead

In [ ]:
import integrate
integrate.main()

## Matrices resultantes

* **Matriz municipal** (125 municipios × ~19 variables): insumo del Módulo A.
* **Matriz de estudiantes** (8.157 × 23 columnas: 15 features candidatas + contexto
  + constructoras del target): insumo del Módulo B.

El log confirma cruces del **100%** en todas las fuentes (meta del concurso: >95%).

In [ ]:
m = pd.read_parquet("../data/processed/matriz_municipal.parquet")
print(f"matriz municipal: {m.shape[0]} municipios x {m.shape[1]} columnas")
m.head(3)

In [ ]:
e = pd.read_parquet("../data/processed/matriz_estudiantes.parquet")
print(f"matriz de estudiantes: {e.shape[0]} filas x {e.shape[1]} columnas")
e.head(3)

In [ ]:
print(open('../outputs/log_integracion.txt', encoding='utf-8').read())